# Project 5: Logistic Regression from Scratch

## Introduction

Implement **Logistic Regression** from scratch using **Gradient Descent** - the foundation of modern deep learning optimizers!

### The Logistic Regression Model

For binary classification (y ∈ {0, 1}):

$$P(y=1|x) = \sigma(w^T x + b) = \frac{1}{1 + e^{-(w^T x + b)}}$$

### Loss Function (Cross-Entropy)

$$L(w) = -\frac{1}{n}\sum_{i=1}^n [y_i \log(\hat{y}_i) + (1-y_i)\log(1-\hat{y}_i)]$$

### Gradient Descent Update

$$w \leftarrow w - \alpha \nabla L(w)$$

Where gradient is:
$$\nabla L(w) = \frac{1}{n}X^T(\hat{y} - y)$$

### Learning Objectives

1. Implement gradient descent from scratch
2. Compare batch vs stochastic GD
3. Visualize decision boundaries
4. Understand convergence behavior
5. Connect to neural network training

---

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import make_classification, make_moons
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression as SklearnLR
from sklearn.metrics import accuracy_score, confusion_matrix
import seaborn as sns

np.random.seed(42)
plt.rcParams['figure.figsize'] = (12, 6)

## Part 1: Implement Logistic Regression

In [ ]:
class LogisticRegression:
    """
    Logistic Regression with Gradient Descent.
    """
    
    def __init__(self, learning_rate=0.01, n_iterations=1000, regularization=0.0):
        self.lr = learning_rate
        self.n_iterations = n_iterations
        self.reg = regularization
        self.weights = None
        self.bias = None
        self.loss_history = []
        
    @staticmethod
    def sigmoid(z):
        """Sigmoid activation function."""
        return 1 / (1 + np.exp(-np.clip(z, -500, 500)))  # Clip for numerical stability
    
    def compute_loss(self, y_true, y_pred):
        """Binary cross-entropy loss with L2 regularization."""
        n = len(y_true)
        epsilon = 1e-15  # For numerical stability
        y_pred = np.clip(y_pred, epsilon, 1 - epsilon)
        
        # Cross-entropy
        ce_loss = -np.mean(y_true * np.log(y_pred) + (1 - y_true) * np.log(1 - y_pred))
        
        # L2 regularization
        reg_loss = (self.reg / 2) * np.sum(self.weights ** 2)
        
        return ce_loss + reg_loss
    
    def fit(self, X, y, verbose=True):
        """Train using batch gradient descent."""
        n_samples, n_features = X.shape
        
        # Initialize parameters
        self.weights = np.zeros(n_features)
        self.bias = 0
        
        # Gradient descent
        for i in range(self.n_iterations):
            # Forward pass
            z = X @ self.weights + self.bias
            y_pred = self.sigmoid(z)
            
            # Compute loss
            loss = self.compute_loss(y, y_pred)
            self.loss_history.append(loss)
            
            # Backward pass (gradients)
            error = y_pred - y
            dw = (1/n_samples) * (X.T @ error) + self.reg * self.weights
            db = (1/n_samples) * np.sum(error)
            
            # Update parameters
            self.weights -= self.lr * dw
            self.bias -= self.lr * db
            
            if verbose and (i+1) % 100 == 0:
                acc = np.mean((y_pred >= 0.5) == y)
                print(f"Iter {i+1}/{self.n_iterations}, Loss: {loss:.4f}, Acc: {acc:.4f}")
    
    def predict_proba(self, X):
        """Predict probabilities."""
        z = X @ self.weights + self.bias
        return self.sigmoid(z)
    
    def predict(self, X):
        """Predict class labels."""
        return (self.predict_proba(X) >= 0.5).astype(int)

print("✓ Logistic Regression implemented!")

## Part 2: Test on Linearly Separable Data

In [ ]:
# Generate data
X, y = make_classification(n_samples=300, n_features=2, n_redundant=0, 
                          n_informative=2, n_clusters_per_class=1, 
                          class_sep=2.0, random_state=42)

# Split and scale
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

# Visualize data
plt.figure(figsize=(8, 6))
plt.scatter(X_train[y_train==0, 0], X_train[y_train==0, 1], 
           c='blue', alpha=0.6, s=50, label='Class 0')
plt.scatter(X_train[y_train==1, 0], X_train[y_train==1, 1], 
           c='red', alpha=0.6, s=50, label='Class 1')
plt.xlabel('Feature 1', fontsize=12)
plt.ylabel('Feature 2', fontsize=12)
plt.title('Training Data', fontsize=14, fontweight='bold')
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

print(f"Training samples: {len(X_train)}")
print(f"Test samples: {len(X_test)}")

In [ ]:
# Train model
model = LogisticRegression(learning_rate=0.1, n_iterations=1000, regularization=0.01)
model.fit(X_train, y_train, verbose=True)

# Evaluate
y_pred_train = model.predict(X_train)
y_pred_test = model.predict(X_test)

print(f"\nTrain Accuracy: {accuracy_score(y_train, y_pred_train):.4f}")
print(f"Test Accuracy: {accuracy_score(y_test, y_pred_test):.4f}")

## Part 3: Visualize Decision Boundary

In [ ]:
def plot_decision_boundary(model, X, y, title="Decision Boundary"):
    """Plot decision boundary for 2D data."""
    h = 0.02  # Mesh step size
    x_min, x_max = X[:, 0].min() - 0.5, X[:, 0].max() + 0.5
    y_min, y_max = X[:, 1].min() - 0.5, X[:, 1].max() + 0.5
    xx, yy = np.meshgrid(np.arange(x_min, x_max, h), np.arange(y_min, y_max, h))
    
    # Predict on mesh
    Z = model.predict(np.c_[xx.ravel(), yy.ravel()])
    Z = Z.reshape(xx.shape)
    
    # Plot
    plt.figure(figsize=(10, 7))
    plt.contourf(xx, yy, Z, alpha=0.3, cmap='RdBu_r')
    plt.scatter(X[y==0, 0], X[y==0, 1], c='blue', edgecolors='k', 
               s=50, label='Class 0', alpha=0.7)
    plt.scatter(X[y==1, 0], X[y==1, 1], c='red', edgecolors='k', 
               s=50, label='Class 1', alpha=0.7)
    plt.xlabel('Feature 1', fontsize=12)
    plt.ylabel('Feature 2', fontsize=12)
    plt.title(title, fontsize=14, fontweight='bold')
    plt.legend()
    plt.grid(alpha=0.3)
    plt.tight_layout()
    plt.show()

plot_decision_boundary(model, X_test, y_test, "Logistic Regression Decision Boundary")

## Part 4: Visualize Training Process

In [ ]:
# Plot loss curve
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 5))

# Loss curve
ax1.plot(model.loss_history, linewidth=2)
ax1.set_xlabel('Iteration', fontsize=12)
ax1.set_ylabel('Loss', fontsize=12)
ax1.set_title('Training Loss Curve', fontsize=14, fontweight='bold')
ax1.grid(alpha=0.3)

# Loss curve (zoomed)
ax2.plot(model.loss_history[100:], linewidth=2)
ax2.set_xlabel('Iteration', fontsize=12)
ax2.set_ylabel('Loss', fontsize=12)
ax2.set_title('Training Loss (After Iteration 100)', fontsize=14, fontweight='bold')
ax2.grid(alpha=0.3)

plt.tight_layout()
plt.show()

print(f"Initial loss: {model.loss_history[0]:.4f}")
print(f"Final loss: {model.loss_history[-1]:.4f}")
print(f"Improvement: {(model.loss_history[0] - model.loss_history[-1]):.4f}")

## Part 5: Stochastic Gradient Descent

In [ ]:
class LogisticRegressionSGD:
    """Logistic Regression with Stochastic Gradient Descent."""
    
    def __init__(self, learning_rate=0.01, n_epochs=100, batch_size=32, regularization=0.0):
        self.lr = learning_rate
        self.n_epochs = n_epochs
        self.batch_size = batch_size
        self.reg = regularization
        self.weights = None
        self.bias = None
        self.loss_history = []
        
    @staticmethod
    def sigmoid(z):
        return 1 / (1 + np.exp(-np.clip(z, -500, 500)))
    
    def fit(self, X, y, verbose=True):
        n_samples, n_features = X.shape
        self.weights = np.zeros(n_features)
        self.bias = 0
        
        n_batches = n_samples // self.batch_size
        
        for epoch in range(self.n_epochs):
            # Shuffle data
            indices = np.random.permutation(n_samples)
            X_shuffled = X[indices]
            y_shuffled = y[indices]
            
            epoch_loss = 0
            for i in range(n_batches):
                # Get mini-batch
                start = i * self.batch_size
                end = start + self.batch_size
                X_batch = X_shuffled[start:end]
                y_batch = y_shuffled[start:end]
                
                # Forward pass
                z = X_batch @ self.weights + self.bias
                y_pred = self.sigmoid(z)
                
                # Loss
                epsilon = 1e-15
                y_pred = np.clip(y_pred, epsilon, 1 - epsilon)
                loss = -np.mean(y_batch * np.log(y_pred) + 
                               (1 - y_batch) * np.log(1 - y_pred))
                epoch_loss += loss
                
                # Gradients
                error = y_pred - y_batch
                dw = (1/self.batch_size) * (X_batch.T @ error) + self.reg * self.weights
                db = (1/self.batch_size) * np.sum(error)
                
                # Update
                self.weights -= self.lr * dw
                self.bias -= self.lr * db
            
            epoch_loss /= n_batches
            self.loss_history.append(epoch_loss)
            
            if verbose and (epoch+1) % 10 == 0:
                y_pred_all = self.sigmoid(X @ self.weights + self.bias)
                acc = np.mean((y_pred_all >= 0.5) == y)
                print(f"Epoch {epoch+1}/{self.n_epochs}, Loss: {epoch_loss:.4f}, Acc: {acc:.4f}")
    
    def predict_proba(self, X):
        z = X @ self.weights + self.bias
        return self.sigmoid(z)
    
    def predict(self, X):
        return (self.predict_proba(X) >= 0.5).astype(int)

# Train SGD model
model_sgd = LogisticRegressionSGD(learning_rate=0.1, n_epochs=100, batch_size=16, regularization=0.01)
model_sgd.fit(X_train, y_train, verbose=True)

# Evaluate
y_pred_test_sgd = model_sgd.predict(X_test)
print(f"\nTest Accuracy (SGD): {accuracy_score(y_test, y_pred_test_sgd):.4f}")

## Part 6: Compare Batch GD vs SGD

In [ ]:
# Train fresh models for comparison
model_batch = LogisticRegression(learning_rate=0.1, n_iterations=100, regularization=0.01)
model_batch.fit(X_train, y_train, verbose=False)

model_sgd_cmp = LogisticRegressionSGD(learning_rate=0.1, n_epochs=100, batch_size=16, regularization=0.01)
model_sgd_cmp.fit(X_train, y_train, verbose=False)

# Plot comparison
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 5))

# Loss curves
ax1.plot(model_batch.loss_history, label='Batch GD', linewidth=2)
ax1.plot(model_sgd_cmp.loss_history, label='SGD (batch=16)', linewidth=2, alpha=0.7)
ax1.set_xlabel('Iteration/Epoch', fontsize=12)
ax1.set_ylabel('Loss', fontsize=12)
ax1.set_title('Convergence: Batch GD vs SGD', fontsize=14, fontweight='bold')
ax1.legend()
ax1.grid(alpha=0.3)

# Decision boundaries comparison
h = 0.02
x_min, x_max = X_test[:, 0].min() - 0.5, X_test[:, 0].max() + 0.5
y_min, y_max = X_test[:, 1].min() - 0.5, X_test[:, 1].max() + 0.5
xx, yy = np.meshgrid(np.arange(x_min, x_max, h), np.arange(y_min, y_max, h))

Z_batch = model_batch.predict_proba(np.c_[xx.ravel(), yy.ravel()])
Z_batch = Z_batch.reshape(xx.shape)

ax2.contourf(xx, yy, Z_batch, alpha=0.3, levels=20, cmap='RdBu_r')
ax2.contour(xx, yy, Z_batch, levels=[0.5], colors='black', linewidths=2, label='Decision boundary')
ax2.scatter(X_test[y_test==0, 0], X_test[y_test==0, 1], c='blue', 
           edgecolors='k', s=50, label='Class 0', alpha=0.7)
ax2.scatter(X_test[y_test==1, 0], X_test[y_test==1, 1], c='red', 
           edgecolors='k', s=50, label='Class 1', alpha=0.7)
ax2.set_xlabel('Feature 1', fontsize=12)
ax2.set_ylabel('Feature 2', fontsize=12)
ax2.set_title('Final Decision Boundary', fontsize=14, fontweight='bold')
ax2.legend()
ax2.grid(alpha=0.3)

plt.tight_layout()
plt.show()

print("\n=== Comparison ===")
print(f"Batch GD - Final loss: {model_batch.loss_history[-1]:.4f}, Test Acc: {accuracy_score(y_test, model_batch.predict(X_test)):.4f}")
print(f"SGD - Final loss: {model_sgd_cmp.loss_history[-1]:.4f}, Test Acc: {accuracy_score(y_test, model_sgd_cmp.predict(X_test)):.4f}")

## Part 7: Compare with Scikit-Learn

In [ ]:
# Train sklearn model
sklearn_model = SklearnLR(max_iter=1000, random_state=42)
sklearn_model.fit(X_train, y_train)
y_pred_sklearn = sklearn_model.predict(X_test)

print("=== Performance Comparison ===")
print(f"Our Batch GD:  {accuracy_score(y_test, model.predict(X_test)):.4f}")
print(f"Our SGD:       {accuracy_score(y_test, model_sgd.predict(X_test)):.4f}")
print(f"Scikit-learn:  {accuracy_score(y_test, y_pred_sklearn):.4f}")

# Confusion matrix
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

for ax, preds, title in zip(axes, 
                            [model.predict(X_test), model_sgd.predict(X_test), y_pred_sklearn],
                            ['Our Batch GD', 'Our SGD', 'Scikit-learn']):
    cm = confusion_matrix(y_test, preds)
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax, cbar=False)
    ax.set_title(title, fontsize=12, fontweight='bold')
    ax.set_xlabel('Predicted')
    ax.set_ylabel('Actual')

plt.tight_layout()
plt.show()

## Summary

### What We Learned

1. **Logistic regression** uses sigmoid activation for binary classification
2. **Cross-entropy loss** is convex → gradient descent finds global optimum
3. **Batch GD** uses all data → smooth convergence
4. **SGD** uses mini-batches → faster, noisier, better generalization
5. **Learning rate** is crucial: too small = slow, too large = divergence

### Connection to Deep Learning

Logistic regression is a **single-layer neural network**:
- Input → Linear (w^T x + b) → Sigmoid → Output
- Same gradient descent, same loss function
- Deep learning = stack many layers!

Modern optimizers extend SGD:
- **Momentum**: Add velocity term for faster convergence
- **Adam**: Adaptive learning rates per parameter
- **RMSprop**: Scale learning rates by gradient magnitudes

### When to Use Logistic Regression

✅ **Good for:**
- Binary classification
- Interpretable coefficients
- Fast training
- Baseline model

❌ **Limitations:**
- Only linear decision boundaries
- Can't capture complex patterns
- Use neural networks for non-linear problems

### Next Steps

- **Project 6**: Portfolio optimization with quadratic programming
- **Project 7**: Compressed sensing - the ultimate L1 application
- Explore neural networks (multi-layer logistic regression!)

---

**Awesome!** You've implemented the foundation of modern ML from scratch! 🚀